# Flow-Aware Temporal Pattern Mining for Multi-Stage Network Intrusion Detection

13-stage NIDS pipeline on CIC-IDS2017, one stage per cell, exactly as specified:
data loading -> dual-branch imbalance handling -> feature engineering ->
bidirectional session reconstruction -> behavioral event encoding -> parallel
RF/BiLSTM/XGBoost training -> adaptive evidence fusion -> FP-Growth ->
PrefixSpan -> temporal attack-state graph -> adaptive risk scoring ->
explainability -> evaluation.

**Dataset:** `D:\IDSPROJECT2026\CIC-IDS2017` (edit `DATA_DIR` in Cell 1).

**A few places where the literal spec needed an engineering decision** (flagged
inline with a comment at the point it matters, not buried here):

- **XGBoost `scale_pos_weight`** is a *binary*-classification parameter (the
  ratio for the single positive class); it has no multi-class equivalent.
  The multi-class analogue of "apply Branch A class weights to XGBoost" is
  `sample_weight` in `.fit()`, which is what Cell 7 uses instead.
- **Event-encoding rule priority.** The spec gives each of the 10 tokens'
  *trigger conditions* but not an evaluation order, and several conditions
  overlap (e.g. an auth-port flow that ends in RST matches both `AUTH_FAIL`
  and `FORCED_TERMINATION`). Cell 6 evaluates the most specific/severe
  conditions first (DoS-rate, scan fan-out, auth outcome) before the generic
  handshake/termination catch-alls, with the exact order documented in the
  cell. `POST_AUTH_ACTIVITY` is inherently sequence-dependent ("follows
  AUTH_SUCCESS"), so it is applied as a second pass over each session's
  ordered tokens, not a per-flow rule.
- **Branch A vs Branch B, for the Stage 13 McNemar comparison to be
  meaningful,** Stage 6 trains RF *twice* (once class-weighted on original
  data = Branch A, once on SMOTE+ENN-resampled data with no extra weighting
  = Branch B) and XGBoost likewise. The Branch A models feed the main
  fusion/risk/evaluation pipeline (Stages 7-13); Branch B exists for the
  Stage 2 demonstration and the Stage 13 ablation comparison.
- **`G_weight_t`** in Stage 11's risk formula is used in Stage 10 but never
  formally defined in the spec; it is implemented as the mean raw
  (non-time-decayed) edge weight along the session's token path -- the
  purely structural companion to the time-decayed `TC_t`.


In [ ]:
# ============================================================
# CELL 1 -- Imports and global setup
# ============================================================
import os
import re
import glob
import random
import warnings
from pathlib import Path

try:
    import numpy as np
    import pandas as pd

    warnings.filterwarnings("ignore")
    RANDOM_SEED = 42
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    pd.set_option("display.max_columns", 60)
    pd.set_option("display.width", 160)

    # --- EDIT THIS if your dataset lives somewhere else ----------------------
    DATA_DIR = r"D:\IDSPROJECT2026\CIC-IDS2017"
    # --------------------------------------------------------------------------

    # The 15 canonical CIC-IDS2017 classes (K=15). Heartbleed and Infiltration
    # are always reported per-class, never merged into a rare-class bucket.
    CLASSES = [
        "BENIGN", "DoS Hulk", "PortScan", "DDoS", "DoS GoldenEye",
        "FTP-Patator", "SSH-Patator", "DoS slowloris", "DoS Slowhttptest",
        "Bot", "Web Attack - Brute Force", "Web Attack - XSS",
        "Web Attack - Sql Injection", "Infiltration", "Heartbleed",
    ]
    K = len(CLASSES)  # 15
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

    # 10-token behavioral vocabulary (Stage 5).
    TOKENS = [
        "CONNECTION_ATTEMPT", "SESSION_ESTABLISHED", "AUTH_FAIL", "AUTH_SUCCESS",
        "SCAN_ACTIVITY", "DATA_TRANSFER", "CONNECTION_TERMINATION",
        "FORCED_TERMINATION", "DOS_INDICATOR", "POST_AUTH_ACTIVITY",
    ]
    TOKEN_TO_IDX = {t: i for i, t in enumerate(TOKENS)}
    AUTH_PORTS = {22, 23, 3389, 21}

    MODELS_DIR = Path("./models")
    MODELS_DIR.mkdir(exist_ok=True)

    print("Setup OK. RANDOM_SEED =", RANDOM_SEED)
    print(f"DATA_DIR = {DATA_DIR}")
    print(f"K = {K} classes:", CLASSES)
    print(f"Token vocabulary ({len(TOKENS)}):", TOKENS)

except Exception as e:
    print(f"[Cell 1 ERROR] Setup failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 2 -- Stage 1: Data Loading & Chronological Split
# ============================================================
import glob
import re
import pandas as pd
import numpy as np

try:
    # Discover every CSV under DATA_DIR and assign a Day_Index (1-5) by
    # matching the weekday name in the filename -- robust to whether
    # Thursday/Friday ship as 1 or 2 physical files each (both files for a
    # given weekday get the SAME Day_Index, since the split operates on
    # calendar day, not physical file count).
    DAY_INDEX = {"monday": 1, "tuesday": 2, "wednesday": 3, "thursday": 4, "friday": 5}

    csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv"))) + \
                sorted(glob.glob(os.path.join(DATA_DIR, "*.CSV")))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under {DATA_DIR}")

    frames = []
    for f in csv_files:
        low = os.path.basename(f).lower()
        day_idx = next((idx for name, idx in DAY_INDEX.items() if name in low), None)
        if day_idx is None:
            print(f"  [skip] could not infer weekday for {os.path.basename(f)}")
            continue
        df_f = pd.read_csv(f, low_memory=False)
        df_f.columns = df_f.columns.str.strip()  # strip whitespace from column names
        df_f["Day_Index"] = day_idx
        df_f["Source_File"] = os.path.basename(f)
        frames.append(df_f)
        print(f"  loaded {os.path.basename(f)}: {len(df_f):,} rows -> Day_Index={day_idx}")

    df_all = pd.concat(frames, axis=0, ignore_index=True, sort=False)

    # Map ALL benign label spellings to a single canonical 'BENIGN'.
    df_all["Label"] = df_all["Label"].astype(str).str.strip()
    is_benign = df_all["Label"].str.lower().str.contains("benign")
    df_all.loc[is_benign, "Label"] = "BENIGN"
    # Harmonize the remaining known spelling/encoding variants across daily files.
    label_fix = {
        "Web Attack \x96 Brute Force": "Web Attack - Brute Force",
        "Web Attack \x96 XSS": "Web Attack - XSS",
        "Web Attack \x96 Sql Injection": "Web Attack - Sql Injection",
        "Web Attack – Brute Force": "Web Attack - Brute Force",
        "Web Attack – XSS": "Web Attack - XSS",
        "Web Attack – Sql Injection": "Web Attack - Sql Injection",
    }
    df_all["Label"] = df_all["Label"].replace(label_fix)

    # CHRONOLOGICAL split -- NO random shuffle. Days 1-2 = Train, Day 3 = Val,
    # Days 4-5 = Test. This is a hard requirement: a random split would place
    # flows from the same attack session in both train and test, making any
    # later session-level / temporal evaluation trivially optimistic.
    train_df = df_all[df_all["Day_Index"].isin([1, 2])].reset_index(drop=True)
    val_df = df_all[df_all["Day_Index"] == 3].reset_index(drop=True)
    test_df = df_all[df_all["Day_Index"].isin([4, 5])].reset_index(drop=True)

    print(f"\nTotal flows: {len(df_all):,}")
    print(f"Train (Days 1-2): {len(train_df):,} | Val (Day 3): {len(val_df):,} | Test (Days 4-5): {len(test_df):,}")

    print("\nClass distribution -- TRAIN:")
    print(train_df["Label"].value_counts())
    print("\nClass distribution -- VAL:")
    print(val_df["Label"].value_counts())
    print("\nClass distribution -- TEST:")
    print(test_df["Label"].value_counts())

    assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0, "one or more splits is empty"

except FileNotFoundError as e:
    print(f"[Cell 2 ERROR] {e}")
    print("Check that DATA_DIR (Cell 1) points at the folder containing the CIC-IDS2017 daily CSVs.")
    raise
except Exception as e:
    print(f"[Cell 2 ERROR] Stage 1 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 3 -- Stage 2: Dual-Branch Class Imbalance Handling
# ============================================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours

try:
    # --- Branch A: Class Weighting (applied to ALL models) -------------------
    # W_c = N_train / (K * N_c). A class absent from training gets weight 0
    # (it contributes nothing to any loss -- see the Heartbleed/Infiltration
    # zero-training-count note below).
    N_train = len(train_df)
    train_counts = train_df["Label"].value_counts()
    class_weights_A = {
        c: (N_train / (K * train_counts[c])) if c in train_counts and train_counts[c] > 0 else 0.0
        for c in CLASSES
    }
    print("Branch A class weights W_c = N_train / (K * N_c):")
    print(pd.Series(class_weights_A).sort_values(ascending=False))

    zero_train_classes = [c for c, w in class_weights_A.items() if w == 0.0]
    if zero_train_classes:
        print(f"\n[NOTE] Classes with ZERO training examples: {zero_train_classes}. "
              "In the real CIC-IDS2017 release, Heartbleed appears only in the "
              "Wednesday capture and Infiltration only in Thursday -- under this "
              "chronological split they are legitimately absent from training. "
              "Report this honestly rather than as a bug.")

    # --- Branch B: SMOTE + Edited Nearest Neighbours (RF/XGBoost ONLY) -------
    # Skip classes with N_c < 50 (Heartbleed N=11, Infiltration N=36) -- SMOTE
    # with k=5 neighbours on <50 samples just re-interpolates the same handful
    # of points repeatedly and adds no real information.
    # NOTE ON ORDERING: SMOTE must run on SCALED features (distances are
    # meaningless on raw CICFlowMeter columns, where Flow Bytes/s can be in
    # the millions next to a 0/1 flag column). Stage 3 (next cell) is where
    # the real, per-model-group scaling happens, so the REAL Branch B
    # resampling used for RF/XGBoost training happens in Stage 6, on the
    # correctly scaled Group A+B / Group B+D matrices respectively (RF and
    # XGBoost consume different feature groups, so each needs its own
    # resample). This cell demonstrates the mechanism now, on a quick
    # temporary standard-scaled numeric feature matrix, purely to print the
    # required before/after class counts.
    numeric_cols_preview = train_df.select_dtypes(include=[np.number]).columns
    numeric_cols_preview = [c for c in numeric_cols_preview if c not in ("Day_Index",)]
    X_preview = train_df[numeric_cols_preview].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X_preview_scaled = StandardScaler().fit_transform(X_preview)
    y_preview = train_df["Label"]

    eligible_classes = {c: n for c, n in train_counts.items() if n >= 50 and c != train_counts.idxmax()}
    skipped_classes = {c: n for c, n in train_counts.items() if n < 50}
    print(f"\nBranch B (SMOTE-KNN via SMOTE+ENN) -- skipping classes with N_c<50: {skipped_classes}")

    if eligible_classes:
        majority_n = int(train_counts.max())
        smote_targets = {c: min(int(majority_n * 0.10), int(n * 10)) for c, n in eligible_classes.items()}
        smote_targets = {c: t for c, t in smote_targets.items() if t > eligible_classes[c]}
        print("Before SMOTE (preview):", dict(train_counts))
        if smote_targets:
            sm = SMOTE(sampling_strategy=smote_targets, k_neighbors=5, random_state=RANDOM_SEED)
            X_sm, y_sm = sm.fit_resample(X_preview_scaled, y_preview)
            print("After SMOTE (preview):", pd.Series(y_sm).value_counts().to_dict())

            enn = EditedNearestNeighbours(n_neighbors=3)
            X_smenn, y_smenn = enn.fit_resample(X_sm, y_sm)
            print("After SMOTE+ENN (preview):", pd.Series(y_smenn).value_counts().to_dict())
        else:
            print("No class exceeded its SMOTE target -- nothing to oversample in this preview.")
    else:
        print("No classes eligible for SMOTE-KNN (all minority classes have N_c < 50).")

except Exception as e:
    print(f"[Cell 3 ERROR] Stage 2 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 4 -- Stage 3: Feature Engineering & Grouping
# ============================================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

try:
    # Keyword-matched group assignment against the 78 CICFlowMeter columns
    # (post whitespace-strip from Stage 1). Matching is substring-based and
    # case-insensitive so it tolerates the minor column-naming differences
    # between CIC-IDS2017 distributions.
    GROUP_KEYWORDS = {
        # Group A (~32): flow-statistical features
        "A": ["Flow Duration", "Total Fwd Packets", "Total Backward Packets",
              "Length of Fwd Packets", "Length of Bwd Packets", "Packet Length",
              "Flow Bytes", "Flow Packets", "Packets/s", "Down/Up Ratio",
              "Average Packet Size", "Segment Size", "Subflow", "Bulk"],
        # Group B (~18): protocol / header / non-flag-count communication features
        "B": ["Protocol", "Header Length", "Init_Win_bytes", "act_data_pkt_fwd",
              "min_seg_size_forward", "PSH Flags", "URG Flags"],
        # Group C (~15): derived temporal features (used for Stage 5 rule
        # thresholds on RAW values, not fed directly to a scaled classifier)
        "C": ["IAT Mean", "IAT Std", "IAT Max", "IAT Min", "IAT Total",
              "Active Mean", "Active Std", "Active Max", "Active Min",
              "Idle Mean", "Idle Std", "Idle Max", "Idle Min"],
        # Group D (~13): individual TCP flag counts
        "D": ["FIN Flag Count", "SYN Flag Count", "RST Flag Count", "PSH Flag Count",
              "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count"],
    }
    NON_FEATURE_COLS = {"Flow ID", "Source IP", "Destination IP", "Source Port",
                         "Timestamp", "Label", "Day_Index", "Source_File"}
    numeric_candidates = [c for c in train_df.columns
                           if c not in NON_FEATURE_COLS and pd.api.types.is_numeric_dtype(train_df[c])]

    def assign_group(col):
        for g in ["D", "C", "B", "A"]:  # most-specific keyword sets checked first
            if any(kw.lower() in col.lower() for kw in GROUP_KEYWORDS[g]):
                return g
        return "A"  # fallback -- keeps every numeric column usable, never silently dropped

    groups = {"A": [], "B": [], "C": [], "D": []}
    for c in numeric_candidates:
        groups[assign_group(c)].append(c)
    print("Feature group sizes:", {g: len(v) for g, v in groups.items()})

    # --- Cleaning: Inf -> NaN, drop columns with >90% missing (train-driven) -
    def to_nan_inf(df):
        out = df.copy()
        out[numeric_candidates] = out[numeric_candidates].replace([np.inf, -np.inf], np.nan)
        return out

    train_clean = to_nan_inf(train_df)
    val_clean = to_nan_inf(val_df)
    test_clean = to_nan_inf(test_df)

    missing_frac = train_clean[numeric_candidates].isna().mean()
    drop_cols = missing_frac[missing_frac > 0.90].index.tolist()
    if drop_cols:
        print(f"Dropping {len(drop_cols)} columns with >90% missing (train): {drop_cols}")
        numeric_candidates = [c for c in numeric_candidates if c not in drop_cols]
        for g in groups:
            groups[g] = [c for c in groups[g] if c not in drop_cols]

    # --- Median imputation, fit on TRAIN only ---------------------------------
    train_medians = train_clean[numeric_candidates].median()
    for df_ in (train_clean, val_clean, test_clean):
        df_[numeric_candidates] = df_[numeric_candidates].fillna(train_medians)

    # --- Outlier clipping at the 1st-99th percentile, bounds fit on TRAIN -----
    lower_b = train_clean[numeric_candidates].quantile(0.01)
    upper_b = train_clean[numeric_candidates].quantile(0.99)
    for df_ in (train_clean, val_clean, test_clean):
        df_[numeric_candidates] = df_[numeric_candidates].clip(lower=lower_b, upper=upper_b, axis=1)

    # --- StandardScaler on Groups A+B+D, fit on TRAIN only ---------------------
    # Group C is intentionally left UNSCALED: Stage 5's event-encoding rules
    # threshold Group-C-derived quantities (flow duration, IAT) against real
    # physical units (seconds, packets/sec), so those raw values are needed,
    # not a z-scored version of them.
    scale_cols = groups["A"] + groups["B"] + groups["D"]
    scaler = StandardScaler()
    scaler.fit(train_clean[scale_cols])

    def apply_scaler(df_):
        out = df_.copy()
        out[scale_cols] = scaler.transform(df_[scale_cols])
        return out

    train_scaled = apply_scaler(train_clean)
    val_scaled = apply_scaler(val_clean)
    test_scaled = apply_scaler(test_clean)

    print(f"\nScaled columns (A+B+D): {len(scale_cols)}")
    print(f"train_scaled shape: {train_scaled.shape}, val_scaled: {val_scaled.shape}, test_scaled: {test_scaled.shape}")
    train_scaled[scale_cols].describe().loc[["mean", "std"]].iloc[:, :6]

except Exception as e:
    print(f"[Cell 4 ERROR] Stage 3 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 5 -- Stage 4: Bidirectional Session Reconstruction
# ============================================================
import ipaddress
import numpy as np
import pandas as pd

try:
    def find_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        raise KeyError(f"none of {candidates} found in columns")

    SRC_IP_COL = find_col(train_df, ["Source IP", "Src IP"])
    DST_IP_COL = find_col(train_df, ["Destination IP", "Dst IP"])
    SRC_PORT_COL = find_col(train_df, ["Source Port", "Src Port"])
    DST_PORT_COL = find_col(train_df, ["Destination Port", "Dst Port"])
    PROTO_COL = "Protocol"
    TS_COL = "Timestamp"

    def ip_to_int(series):
        def conv(v):
            try:
                return int(ipaddress.ip_address(str(v).strip()))
            except ValueError:
                return abs(hash(str(v))) % (2 ** 32)
        return series.map(conv)

    def build_session_key(df):
        src_ip, dst_ip = ip_to_int(df[SRC_IP_COL]), ip_to_int(df[DST_IP_COL])
        ip_lo, ip_hi = np.minimum(src_ip, dst_ip), np.maximum(src_ip, dst_ip)
        src_p, dst_p = df[SRC_PORT_COL].astype(np.int64), df[DST_PORT_COL].astype(np.int64)
        port_lo, port_hi = np.minimum(src_p, dst_p), np.maximum(src_p, dst_p)
        proto = df[PROTO_COL].astype(str) if PROTO_COL in df.columns else "0"
        return (ip_lo.astype(str) + "_" + ip_hi.astype(str) + "_" + proto
                + "_" + port_lo.astype(str) + "_" + port_hi.astype(str))

    TAU_SECONDS = 60.0  # tau: idle timeout that ends a session

    def reconstruct_sessions(df, split_name):
        df = df.copy()
        df["session_key"] = build_session_key(df)
        if TS_COL in df.columns:
            df[TS_COL] = pd.to_datetime(df[TS_COL], errors="coerce")
            if df[TS_COL].isna().any():
                # fall back to a synthetic monotonic clock for unparsable rows
                fallback = pd.to_datetime(df["Day_Index"] * 86400 + np.arange(len(df)), unit="s")
                df[TS_COL] = df[TS_COL].fillna(pd.Series(fallback, index=df.index))
        else:
            df[TS_COL] = pd.to_datetime(df["Day_Index"] * 86400 + np.arange(len(df)), unit="s")

        order = df.sort_values(["Day_Index", "session_key", TS_COL]).index
        df = df.loc[order].reset_index(drop=True)

        group_keys = (df["Day_Index"].astype(str) + "||" + df["session_key"]).values
        ts = df[TS_COL].values

        session_local_id = np.zeros(len(df), dtype=np.int64)
        current_group, counter, last_ts = None, -1, None
        for i in range(len(df)):
            g = group_keys[i]
            if g != current_group:
                current_group, counter = g, 0
            else:
                gap = (ts[i] - last_ts) / np.timedelta64(1, "s")
                if gap > TAU_SECONDS:
                    counter += 1
            session_local_id[i] = counter
            last_ts = ts[i]

        df["session_id"] = group_keys + "__S" + pd.Series(session_local_id).astype(str).str.zfill(3)
        n_sessions = df["session_id"].nunique()
        print(f"{split_name}: {len(df):,} flows -> {n_sessions:,} sessions "
              f"(tau={TAU_SECONDS:.0f}s, mean flows/session={len(df)/max(n_sessions,1):.2f})")
        return df

    train_scaled = reconstruct_sessions(train_scaled, "TRAIN")
    val_scaled = reconstruct_sessions(val_scaled, "VAL")
    test_scaled = reconstruct_sessions(test_scaled, "TEST")

    example_sid = train_scaled["session_id"].value_counts().idxmax()
    print(f"\nExample session '{example_sid}' ({(train_scaled['session_id']==example_sid).sum()} flows):")
    train_scaled.loc[train_scaled["session_id"] == example_sid, [TS_COL, DST_PORT_COL, "Label"]].head()

except Exception as e:
    print(f"[Cell 5 ERROR] Stage 4 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 6 -- Stage 5: Behavioral Event Vocabulary (Token Encoding)
# ============================================================
import numpy as np
import pandas as pd

try:
    def find_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        raise KeyError(f"none of {candidates} found in columns")

    FWD_PKTS_COL = find_col(train_scaled, ["Total Fwd Packets"])
    BWD_PKTS_COL = find_col(train_scaled, ["Total Backward Packets"])
    FWD_BYTES_COL = find_col(train_scaled, ["Total Length of Fwd Packets"])
    BWD_BYTES_COL = find_col(train_scaled, ["Total Length of Bwd Packets"])
    DUR_COL = find_col(train_scaled, ["Flow Duration"])  # microseconds
    SYN_COL, ACK_COL = "SYN Flag Count", "ACK Flag Count"
    FIN_COL, RST_COL = "FIN Flag Count", "RST Flag Count"
    DST_PORT_COL = find_col(train_scaled, ["Destination Port", "Dst Port"])

    def encode_events(df):
        """Rule-based token assignment. Uses RAW (unscaled) Group C/flag/port
        values -- Stage 3 deliberately left these columns unscaled for exactly
        this purpose. Priority order (most specific/severe first, since several
        of the spec's conditions can co-fire on the same flow):
          1. DOS_INDICATOR      -- extreme packet/byte RATE is the most
                                    distinctive signal; checked first so a
                                    flood is never mis-tagged as a mundane
                                    attempt/termination.
          2. SCAN_ACTIVITY      -- low-packet-count + wide destination-port
                                    fan-out within the session.
          3. AUTH_FAIL / AUTH_SUCCESS -- auth-port outcome, checked before the
                                    generic termination/handshake rules so an
                                    auth failure ending in RST is correctly
                                    tagged AUTH_FAIL, not FORCED_TERMINATION.
          4. FORCED_TERMINATION / CONNECTION_TERMINATION -- generic RST/FIN close.
          5. SESSION_ESTABLISHED -- generic completed handshake.
          6. DATA_TRANSFER       -- substantial bidirectional exchange.
          7. CONNECTION_ATTEMPT  -- fallback: an unanswered SYN.
        POST_AUTH_ACTIVITY is NOT in this per-flow pass -- it is inherently
        sequence-dependent ("follows AUTH_SUCCESS") and is applied as a
        second pass below, per session, in chronological order.
        """
        df = df.copy()
        total_pkts = df[FWD_PKTS_COL] + df[BWD_PKTS_COL]
        total_bytes = df[FWD_BYTES_COL] + df[BWD_BYTES_COL]
        duration_s = (df[DUR_COL].clip(lower=0) / 1e6)  # microseconds -> seconds
        pps = total_pkts / duration_s.replace(0, np.nan)
        bps = total_bytes / duration_s.replace(0, np.nan)
        bytes_per_pkt = total_bytes / total_pkts.replace(0, np.nan)
        syn = df[SYN_COL] if SYN_COL in df.columns else pd.Series(0, index=df.index)
        ack = df[ACK_COL] if ACK_COL in df.columns else pd.Series(0, index=df.index)
        fin = df[FIN_COL] if FIN_COL in df.columns else pd.Series(0, index=df.index)
        rst = df[RST_COL] if RST_COL in df.columns else pd.Series(0, index=df.index)
        is_auth_port = df[DST_PORT_COL].isin(AUTH_PORTS)

        # session-level unique destination-port fan-out, merged back to flow level
        unique_dst_ports = df.groupby("session_id")[DST_PORT_COL].transform("nunique")

        token = pd.Series(TOKENS[0], index=df.index, dtype=object)
        assigned = pd.Series(False, index=df.index)

        def assign(mask, tok):
            m = mask.fillna(False) & ~assigned
            token[m] = tok
            assigned[m] |= m

        assign((pps > 1000) | (bps > 1_000_000), "DOS_INDICATOR")
        assign((total_pkts <= 2) & (unique_dst_ports > 5), "SCAN_ACTIVITY")
        assign(is_auth_port & (duration_s < 2) & ((fin > 0) | (rst > 0)), "AUTH_FAIL")
        assign(is_auth_port & (duration_s >= 2) & (total_bytes > 0), "AUTH_SUCCESS")
        assign(rst > 0, "FORCED_TERMINATION")
        assign((fin > 0) & (rst == 0), "CONNECTION_TERMINATION")
        assign((syn > 0) & (ack > 0), "SESSION_ESTABLISHED")
        assign((bytes_per_pkt > 500) & (duration_s > 1), "DATA_TRANSFER")
        assign((syn > 0) & (ack == 0) & (total_pkts <= 3), "CONNECTION_ATTEMPT")
        assign(pd.Series(True, index=df.index), "CONNECTION_ATTEMPT")  # default fallback

        df["token"] = token
        return df

    train_scaled = encode_events(train_scaled)
    val_scaled = encode_events(val_scaled)
    test_scaled = encode_events(test_scaled)

    def apply_post_auth_override(df):
        """Second pass: any token immediately following AUTH_SUCCESS in the
        same session (chronological order) becomes POST_AUTH_ACTIVITY,
        overriding whatever the per-flow rules assigned it."""
        df = df.sort_values(["session_id", "Timestamp"]).copy()
        prev_token = df.groupby("session_id")["token"].shift(1)
        follows_auth_success = prev_token == "AUTH_SUCCESS"
        df.loc[follows_auth_success, "token"] = "POST_AUTH_ACTIVITY"
        return df

    train_scaled = apply_post_auth_override(train_scaled)
    val_scaled = apply_post_auth_override(val_scaled)
    test_scaled = apply_post_auth_override(test_scaled)

    print("Behavioral token distribution -- TRAIN:")
    print(train_scaled["token"].value_counts().reindex(TOKENS, fill_value=0))

except Exception as e:
    print(f"[Cell 6 ERROR] Stage 5 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 7 -- Stage 6: Parallel Model Training (6A RF, 6B BiLSTM, 6C XGBoost)
# ============================================================
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

try:
    tf.random.set_seed(RANDOM_SEED)

    def align_proba(proba, model_classes, target_classes=CLASSES):
        """Reindex a predict_proba matrix onto the full, fixed CLASSES column
        order, filling 0.0 for any class the model never saw in training."""
        out = np.zeros((proba.shape[0], len(target_classes)))
        col_of = {c: i for i, c in enumerate(model_classes)}
        for j, c in enumerate(target_classes):
            if c in col_of:
                out[:, j] = proba[:, col_of[c]]
        return out

    def smote_enn_resample(X, y, min_class_count=50):
        counts = y.value_counts()
        eligible = {c: n for c, n in counts.items() if n >= min_class_count and c != counts.idxmax()}
        if not eligible:
            return X, y
        majority_n = int(counts.max())
        targets = {c: min(int(majority_n * 0.10), int(n * 10)) for c, n in eligible.items()}
        targets = {c: t for c, t in targets.items() if t > eligible[c]}
        if not targets:
            return X, y
        X_sm, y_sm = SMOTE(sampling_strategy=targets, k_neighbors=5, random_state=RANDOM_SEED).fit_resample(X, y)
        X_res, y_res = EditedNearestNeighbours(n_neighbors=3).fit_resample(X_sm, y_sm)
        return pd.DataFrame(X_res, columns=X.columns), pd.Series(y_res)

    # ----------------------------------------------------------------------
    # 6A -- Random Forest, Groups A+B
    # ----------------------------------------------------------------------
    rf_cols = groups["A"] + groups["B"]
    X_train_AB, y_train_lbl = train_scaled[rf_cols], train_scaled["Label"]
    X_val_AB, X_test_AB = val_scaled[rf_cols], test_scaled[rf_cols]

    RF_PARAMS = dict(n_estimators=200, max_depth=20, min_samples_leaf=5, random_state=RANDOM_SEED, n_jobs=-1)

    # Branch A: class-weighted, trained on ORIGINAL (non-resampled) data --
    # this is the PRIMARY model that feeds Stage 7-13.
    rf_branchA = RandomForestClassifier(class_weight=class_weights_A, **RF_PARAMS)
    rf_branchA.fit(X_train_AB, y_train_lbl)

    # Branch B: SMOTE+ENN-resampled data, no extra class weighting -- kept
    # ONLY for the Stage 2 demonstration and the Stage 13 A-vs-B comparison.
    X_train_AB_res, y_train_res_AB = smote_enn_resample(X_train_AB, y_train_lbl)
    print(f"RF Branch B resample: {len(X_train_AB):,} -> {len(X_train_AB_res):,} rows")
    rf_branchB = RandomForestClassifier(**RF_PARAMS)
    rf_branchB.fit(X_train_AB_res, y_train_res_AB)

    proba_rf_val = align_proba(rf_branchA.predict_proba(X_val_AB), rf_branchA.classes_)
    proba_rf_test = align_proba(rf_branchA.predict_proba(X_test_AB), rf_branchA.classes_)
    proba_rf_test_branchB = align_proba(rf_branchB.predict_proba(X_test_AB), rf_branchB.classes_)
    print("RF (Branch A) trained. Test proba shape:", proba_rf_test.shape)

    # ----------------------------------------------------------------------
    # 6C -- XGBoost, Groups B+D
    # ----------------------------------------------------------------------
    xgb_cols = groups["B"] + groups["D"]
    X_train_BD, X_val_BD, X_test_BD = train_scaled[xgb_cols], val_scaled[xgb_cols], test_scaled[xgb_cols]

    XGB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                       objective="multi:softprob", tree_method="hist", random_state=RANDOM_SEED)

    def fit_xgb(X, y, sample_weight=None):
        # NOTE: `scale_pos_weight` is a BINARY-classification parameter (the
        # ratio for the single positive class) and has no multi-class
        # equivalent -- passing it here would silently be ignored/misapplied.
        # The correct multi-class analogue of "apply Branch A class weights
        # to XGBoost" is `sample_weight` in .fit(), used below instead.
        # XGBoost's sklearn API in this version also requires 0..n-1 integer
        # labels rather than raw class strings, so we encode manually and
        # remember the mapping to align predict_proba back to CLASSES order.
        present = sorted(y.unique())
        label_to_idx = {c: i for i, c in enumerate(present)}
        y_idx = y.map(label_to_idx).values
        model = xgb.XGBClassifier(num_class=len(present), **XGB_PARAMS)
        model.fit(X, y_idx, sample_weight=sample_weight)
        return model, present

    sw_branchA = y_train_lbl.map(class_weights_A).values
    xgb_branchA, xgb_branchA_classes = fit_xgb(X_train_BD, y_train_lbl, sample_weight=sw_branchA)

    X_train_BD_res, y_train_res_BD = smote_enn_resample(X_train_BD, y_train_lbl)
    print(f"XGB Branch B resample: {len(X_train_BD):,} -> {len(X_train_BD_res):,} rows")
    xgb_branchB, xgb_branchB_classes = fit_xgb(X_train_BD_res, y_train_res_BD)

    proba_xgb_val = align_proba(xgb_branchA.predict_proba(X_val_BD), xgb_branchA_classes)
    proba_xgb_test = align_proba(xgb_branchA.predict_proba(X_test_BD), xgb_branchA_classes)
    proba_xgb_test_branchB = align_proba(xgb_branchB.predict_proba(X_test_BD), xgb_branchB_classes)
    print("XGBoost (Branch A) trained. Test proba shape:", proba_xgb_test.shape)

    # ----------------------------------------------------------------------
    # 6B -- BiLSTM on token sequences ONLY (never SMOTE-resampled)
    # ----------------------------------------------------------------------
    def session_table(df):
        """One row per session: ordered token list + the session's label
        (majority vote across its flows, matching how CIC-IDS2017 sessions
        are almost always single-label in practice)."""
        df_sorted = df.sort_values(["session_id", "Timestamp"])
        seqs = df_sorted.groupby("session_id")["token"].apply(list)
        labels = df_sorted.groupby("session_id")["Label"].agg(lambda s: s.value_counts().idxmax())
        return pd.DataFrame({"tokens": seqs, "label": labels}).reset_index()

    train_sessions = session_table(train_scaled)
    val_sessions = session_table(val_scaled)
    test_sessions = session_table(test_scaled)

    def tokens_to_int(seqs):
        # tokens indexed 1..10 (0 reserved for padding, mask_zero=True below)
        return [[TOKEN_TO_IDX[t] + 1 for t in seq] for seq in seqs]

    MAX_LEN = 50
    X_train_seq = pad_sequences(tokens_to_int(train_sessions["tokens"]), maxlen=MAX_LEN, padding="post", truncating="post")
    X_val_seq = pad_sequences(tokens_to_int(val_sessions["tokens"]), maxlen=MAX_LEN, padding="post", truncating="post")
    X_test_seq = pad_sequences(tokens_to_int(test_sessions["tokens"]), maxlen=MAX_LEN, padding="post", truncating="post")

    y_train_seq_idx = train_sessions["label"].map(CLASS_TO_IDX).fillna(0).astype(int).values
    sample_weight_lstm = train_sessions["label"].map(class_weights_A).fillna(0.0).values

    bilstm = Sequential([
        Embedding(input_dim=len(TOKENS) + 1, output_dim=32, mask_zero=True),
        Bidirectional(LSTM(128)),
        Dense(K, activation="softmax"),
    ])
    bilstm.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    bilstm.fit(
        X_train_seq, y_train_seq_idx, sample_weight=sample_weight_lstm,
        epochs=15, batch_size=64, verbose=0,
    )
    print("BiLSTM trained on", len(train_sessions), "ORIGINAL (non-resampled) sessions.")

    proba_lstm_val = bilstm.predict(X_val_seq, verbose=0)
    proba_lstm_test = bilstm.predict(X_test_seq, verbose=0)

    # ----------------------------------------------------------------------
    # Save trained models
    # ----------------------------------------------------------------------
    joblib.dump(rf_branchA, MODELS_DIR / "rf_branchA.joblib")
    joblib.dump(rf_branchB, MODELS_DIR / "rf_branchB.joblib")
    joblib.dump({"model": xgb_branchA, "classes": xgb_branchA_classes}, MODELS_DIR / "xgb_branchA.joblib")
    joblib.dump({"model": xgb_branchB, "classes": xgb_branchB_classes}, MODELS_DIR / "xgb_branchB.joblib")
    # Keras models are not reliably joblib-picklable -- use Keras' own format.
    bilstm.save(MODELS_DIR / "bilstm.keras")
    print(f"\nModels saved to {MODELS_DIR.resolve()}")
    print("Stage 6 complete: RF (A+B), XGBoost (A+B), BiLSTM (A only, never SMOTE) all trained.")

except Exception as e:
    print(f"[Cell 7 ERROR] Stage 6 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 8 -- Stage 7: Adaptive Evidence Fusion
# ============================================================
import numpy as np
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score

try:
    # Aggregate per-flow RF/XGBoost probabilities to SESSION level via mean
    # pooling (grouped by session_id); the BiLSTM already outputs one
    # probability vector per session directly.
    def flow_proba_to_session(df, proba):
        p = pd.DataFrame(proba, columns=CLASSES, index=df.index)
        p["session_id"] = df["session_id"].values
        return p.groupby("session_id")[CLASSES].mean()

    P_A_val_sess = flow_proba_to_session(val_scaled, proba_rf_val).reindex(val_sessions["session_id"]).values
    P_C_val_sess = flow_proba_to_session(val_scaled, proba_xgb_val).reindex(val_sessions["session_id"]).values
    P_B_val_sess = proba_lstm_val  # already session-aligned to val_sessions row order

    P_A_test_sess = flow_proba_to_session(test_scaled, proba_rf_test).reindex(test_sessions["session_id"]).values
    P_C_test_sess = flow_proba_to_session(test_scaled, proba_xgb_test).reindex(test_sessions["session_id"]).values
    P_B_test_sess = proba_lstm_test

    y_val_idx = val_sessions["label"].map(CLASS_TO_IDX).fillna(0).astype(int).values

    # R_t = w_A*P_A + w_B*P_B + w_C*P_C, weights optimised on VALIDATION macro-F1
    # via scipy.optimize, then FROZEN for test (never retuned on test labels).
    # Macro-F1-via-argmax is piecewise-constant, so a gradient method (SLSQP)
    # gets stuck at its starting point; differential_evolution (also
    # scipy.optimize, population-based, derivative-free) is used instead.
    # Optimise 2 free parameters (w_A, w_B); w_C = 1 - w_A - w_B, projected
    # onto the simplex (infeasible points where w_C<0 are penalised).
    def neg_macro_f1(w2):
        w_a, w_b = w2
        w_c = 1.0 - w_a - w_b
        if w_c < 0:
            return 1.0
        R = w_a * P_A_val_sess + w_b * P_B_val_sess + w_c * P_C_val_sess
        return -f1_score(y_val_idx, R.argmax(1), average="macro", zero_division=0)

    result = differential_evolution(neg_macro_f1, bounds=[(0, 1), (0, 1)], seed=RANDOM_SEED, maxiter=60, popsize=20, tol=1e-7)
    w_A, w_B = result.x
    w_C = 1.0 - w_A - w_B
    val_macro_f1_at_weights = -result.fun

    print(f"Frozen fusion weights: w_A={w_A:.4f}, w_B={w_B:.4f}, w_C={w_C:.4f}")
    print(f"Validation macro-F1 at these weights: {val_macro_f1_at_weights:.4f}")

    R_t_test = w_A * P_A_test_sess + w_B * P_B_test_sess + w_C * P_C_test_sess
    print(f"\nFused R_t computed for {len(R_t_test)} test sessions. Shape: {R_t_test.shape}")

except Exception as e:
    print(f"[Cell 8 ERROR] Stage 7 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 9 -- Stage 8: Frequent Pattern Mining (FP-Growth, unordered)
# ============================================================
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

try:
    # Transaction = the SET of unique tokens observed in one TRAINING session
    # (order ignored -- that is what makes this "unordered" co-occurrence
    # mining, complementary to the ORDERED PrefixSpan mining in Stage 9).
    train_transactions = [sorted(set(seq)) for seq in train_sessions["tokens"]]

    te = TransactionEncoder()
    onehot = pd.DataFrame(te.fit(train_transactions).transform(train_transactions), columns=te.columns_)
    frequent_itemsets = fpgrowth(onehot, min_support=0.15, use_colnames=True)
    print(f"Mined {len(frequent_itemsets)} frequent itemsets (min_support=0.15).")

    # Association rules with confidence as the ranking metric. min_threshold=0.5
    # keeps only rules where the consequent follows at least half the time the
    # antecedent itemset is present -- a reasonable "meaningfully predictive"
    # bar for a rule to contribute to the pattern-support score below.
    if not frequent_itemsets.empty:
        rules = association_rules(
            frequent_itemsets, num_itemsets=len(onehot), metric="confidence", min_threshold=0.5
        )
    else:
        rules = pd.DataFrame(columns=["antecedents", "consequents", "confidence"])
    print(f"Mined {len(rules)} association rules (confidence >= 0.5).")
    if not rules.empty:
        print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

    def compute_sp_t(token_set, rules_df):
        """SP_t = mean confidence of rules whose antecedent is a SUBSET of
        this session's token set; 0.0 if no rule matches."""
        if rules_df.empty:
            return 0.0
        matches = [r.confidence for r in rules_df.itertuples() if set(r.antecedents).issubset(token_set)]
        return float(np.mean(matches)) if matches else 0.0

    test_sessions["SP_t"] = [compute_sp_t(set(seq), rules) for seq in test_sessions["tokens"]]
    val_sessions["SP_t"] = [compute_sp_t(set(seq), rules) for seq in val_sessions["tokens"]]

    print("\nSP_t distribution (test sessions):")
    print(test_sessions["SP_t"].describe())

except Exception as e:
    print(f"[Cell 9 ERROR] Stage 8 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 10 -- Stage 9: Sequential Pattern Mining (PrefixSpan, ordered)
# ============================================================
import math
import numpy as np
import pandas as pd
from prefixspan import PrefixSpan

try:
    def session_seq_with_time(df):
        """session_id -> chronologically ordered [(token, Timestamp), ...]."""
        df_sorted = df.sort_values(["session_id", "Timestamp"])
        out = {}
        for sid, g in df_sorted.groupby("session_id"):
            out[sid] = list(zip(g["token"].tolist(), g["Timestamp"].tolist()))
        return out

    train_seqs_time = session_seq_with_time(train_scaled)
    val_seqs_time = session_seq_with_time(val_scaled)
    test_seqs_time = session_seq_with_time(test_scaled)

    MAX_GAP_SECONDS = 30.0  # tight temporal gap constraint for PrefixSpan
    MIN_SUPPORT = 0.10

    def matches_with_gap(pattern, seq, max_gap):
        """Greedy left-to-right subsequence match with a max_gap constraint
        between consecutive MATCHED tokens' timestamps."""
        p_idx, last_time = 0, None
        for tok, ts in seq:
            if p_idx >= len(pattern):
                break
            if tok == pattern[p_idx]:
                if last_time is not None and (ts - last_time).total_seconds() > max_gap:
                    p_idx = 1 if tok == pattern[0] else 0
                    last_time = ts if tok == pattern[0] else None
                    continue
                last_time = ts
                p_idx += 1
        return p_idx >= len(pattern)

    # Pass 1: PrefixSpan mines frequent ORDER-only candidate sequences
    # (ignores timing), on training token sequences only.
    order_only_db = [[tok for tok, _ in train_seqs_time[sid]] for sid in train_seqs_time]
    n_sessions = len(order_only_db)
    min_count = max(1, math.ceil(MIN_SUPPORT * n_sessions))
    ps = PrefixSpan(order_only_db)
    ps.minlen, ps.maxlen = 2, 6
    candidates = ps.frequent(min_count)
    candidates = sorted(candidates, key=lambda cp: cp[0], reverse=True)[:200]
    print(f"PrefixSpan: {len(candidates)} order-only candidate sequences (min_support={MIN_SUPPORT}).")

    # Pass 2: re-verify each candidate against REAL timestamps, keeping only
    # patterns that also satisfy the max_gap=30s temporal constraint.
    mined_patterns = []
    for _, pattern in candidates:
        if len(pattern) < 2:
            continue
        hits = sum(matches_with_gap(pattern, train_seqs_time[sid], MAX_GAP_SECONDS) for sid in train_seqs_time)
        support = hits / n_sessions
        if support >= MIN_SUPPORT:
            mined_patterns.append(pattern)
    print(f"After temporal max_gap={MAX_GAP_SECONDS:.0f}s filtering: {len(mined_patterns)} valid sequential patterns.")
    for p in mined_patterns[:5]:
        print("  ", p)

    def sequence_match_ratio(seq, patterns):
        """Fraction of mined patterns that appear as a gap-constrained
        subsequence in this session's token sequence."""
        if not patterns:
            return 0.0
        hits = sum(matches_with_gap(p, seq, MAX_GAP_SECONDS) for p in patterns)
        return hits / len(patterns)

    test_sessions["seq_match_ratio"] = [sequence_match_ratio(test_seqs_time[sid], mined_patterns) for sid in test_sessions["session_id"]]
    val_sessions["seq_match_ratio"] = [sequence_match_ratio(val_seqs_time[sid], mined_patterns) for sid in val_sessions["session_id"]]

    print("\nSequence match ratio distribution (test sessions):")
    print(test_sessions["seq_match_ratio"].describe())

except Exception as e:
    print(f"[Cell 10 ERROR] Stage 9 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 11 -- Stage 10: Temporal Attack-State Graph
# ============================================================
import numpy as np
import pandas as pd
import networkx as nx

try:
    RHO = 0.9          # EMA decay: 90% historical weight, 10% new evidence
    TC_LAMBDA = 0.1     # fixed temporal-decay constant for TC_t

    # Directed graph over the 10 behavioral tokens. ALL edges start at 0.05
    # (uniform prior -- "new edges start at 0.05") before any training
    # observations; each time a transition src->dst is OBSERVED in a
    # training session, the ENTIRE outgoing row from `src` is EMA-updated
    # (indicator 1.0 for the observed dst, 0.0 for every other candidate),
    # so edge weight converges to the relative frequency with which `src`
    # is followed by each token -- exactly the "historically observed
    # frequency of real state transitions" the formula is meant to capture.
    W = {s: {d: 0.05 for d in TOKENS} for s in TOKENS}
    graph = nx.DiGraph()
    graph.add_nodes_from(TOKENS)

    def observe_transition(src, dst):
        for cand in TOKENS:
            indicator = 1.0 if cand == dst else 0.0
            W[src][cand] = RHO * W[src][cand] + (1 - RHO) * indicator

    n_transitions = 0
    # Process TRAINING sessions in chronological order of first event, from
    # ORIGINAL (never SMOTE-resampled) session data only.
    ordered_sids = sorted(train_seqs_time, key=lambda sid: train_seqs_time[sid][0][1])
    for sid in ordered_sids:
        seq = train_seqs_time[sid]
        for (t1, _), (t2, _) in zip(seq, seq[1:]):
            observe_transition(t1, t2)
            graph.add_edge(t1, t2, weight=W[t1][t2])
            n_transitions += 1
    print(f"Attack-state graph built from {len(ordered_sids)} training sessions ({n_transitions} transitions).")

    def edge_weight(src, dst):
        return W[src][dst]

    def compute_tc_t(seq):
        """TC_t = sum_i[W(s_i->s_i+1) * exp(-lambda*dt_i)] / (n-1)."""
        n = len(seq)
        if n <= 1:
            return 0.5  # neutral default for a single-event session (0/0 undefined)
        total = 0.0
        for (t1, ts1), (t2, ts2) in zip(seq, seq[1:]):
            dt = max((ts2 - ts1).total_seconds(), 0.0)
            total += edge_weight(t1, t2) * np.exp(-TC_LAMBDA * dt)
        return total / (n - 1)

    def compute_g_weight(seq):
        """G_weight_t: mean RAW (non-time-decayed) edge weight along the
        session's path -- the purely structural companion to TC_t, fed
        into Stage 11's risk formula alongside it."""
        n = len(seq)
        if n <= 1:
            return 0.5
        return float(np.mean([edge_weight(t1, t2) for (t1, _), (t2, _) in zip(seq, seq[1:])]))

    test_sessions["TC_t"] = [compute_tc_t(test_seqs_time[sid]) for sid in test_sessions["session_id"]]
    test_sessions["G_weight_t"] = [compute_g_weight(test_seqs_time[sid]) for sid in test_sessions["session_id"]]
    val_sessions["TC_t"] = [compute_tc_t(val_seqs_time[sid]) for sid in val_sessions["session_id"]]
    val_sessions["G_weight_t"] = [compute_g_weight(val_seqs_time[sid]) for sid in val_sessions["session_id"]]

    print("\nSample learned edge weights (AUTH_FAIL -> *):")
    print(pd.Series(W["AUTH_FAIL"]).sort_values(ascending=False).head(5))
    print("\nTC_t distribution (test sessions):")
    print(test_sessions["TC_t"].describe())

except Exception as e:
    print(f"[Cell 11 ERROR] Stage 10 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 12 -- Stage 11: Adaptive Risk Scoring
# ============================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

try:
    # R_t on the VALIDATION sessions (needed to train the meta-learner below;
    # Stage 7 already computed and froze w_A/w_B/w_C, and only used them on
    # test -- this is the one additional place they're applied, to validation
    # data, which is standard practice for fitting a stacking meta-learner).
    R_t_val = w_A * P_A_val_sess + w_B * P_B_val_sess + w_C * P_C_val_sess

    def mean_interevent_gap(seq):
        if len(seq) <= 1:
            return 0.0
        return float(np.mean([(t2 - t1).total_seconds() for (_, t1), (_, t2) in zip(seq, seq[1:])]))

    mean_train_gap = np.mean([
        (ts2 - ts1).total_seconds()
        for seq in train_seqs_time.values()
        for (_, ts1), (_, ts2) in zip(seq, seq[1:])
    ]) or 1.0

    def inv_delta_t_norm(seqs_time, sessions_df):
        eps = 1e-3
        gaps = np.array([mean_interevent_gap(seqs_time[sid]) for sid in sessions_df["session_id"]])
        dt_norm = (gaps + eps) / mean_train_gap
        return 1.0 / dt_norm

    val_sessions["inv_dt_norm"] = inv_delta_t_norm(val_seqs_time, val_sessions)
    test_sessions["inv_dt_norm"] = inv_delta_t_norm(test_seqs_time, test_sessions)

    # Meta-features [R_t, SP_t, TC_t, G_weight_t, 1/dt_norm]. R_t is
    # represented by its max-class probability -- the natural scalar
    # analogue of "how strongly does the fused evidence favour ANY single
    # class" usable without knowing the true class at inference time.
    X_val_meta = np.column_stack([
        R_t_val.max(axis=1), val_sessions["SP_t"].values, val_sessions["TC_t"].values,
        val_sessions["G_weight_t"].values, val_sessions["inv_dt_norm"].values,
    ])
    y_val_is_attack = (val_sessions["label"] != "BENIGN").astype(int).values

    # [alpha, beta, gamma, delta, epsilon] are LEARNED, not hand-picked --
    # a logistic regression meta-learner fit on the VALIDATION split.
    risk_model = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED)
    risk_model.fit(X_val_meta, y_val_is_attack)
    coef_names = ["alpha(R_t)", "beta(SP_t)", "gamma(TC_t)", "delta(G_weight_t)", "epsilon(1/dt_norm)"]
    print("Learned risk coefficients:")
    for name, c in zip(coef_names, risk_model.coef_[0]):
        print(f"  {name}: {c:+.4f}")
    print(f"  intercept: {risk_model.intercept_[0]:+.4f}")

    X_test_meta = np.column_stack([
        R_t_test.max(axis=1), test_sessions["SP_t"].values, test_sessions["TC_t"].values,
        test_sessions["G_weight_t"].values, test_sessions["inv_dt_norm"].values,
    ])
    test_sessions["Risk_t"] = risk_model.predict_proba(X_test_meta)[:, 1]

    def tier(risk):
        if risk >= 0.75:
            return "ATTACK"
        if risk >= 0.35:
            return "SUSPICIOUS"
        return "BENIGN"

    test_sessions["alert_tier"] = test_sessions["Risk_t"].apply(tier)
    test_sessions["predicted_class"] = [CLASSES[i] for i in R_t_test.argmax(1)]

    print("\nAlert tier distribution (test sessions):")
    print(test_sessions["alert_tier"].value_counts())

except Exception as e:
    print(f"[Cell 12 ERROR] Stage 11 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 13 -- Stage 12: Explainability
# ============================================================
import numpy as np
import pandas as pd
import tensorflow as tf
import shap

try:
    def explain_tree_model(model, X_row, feature_names, top_k=5):
        """TreeSHAP for RF/XGBoost -- exact, fast for tree ensembles."""
        explainer = shap.TreeExplainer(model)
        raw = explainer.shap_values(X_row)
        if isinstance(raw, list):
            values = raw[int(np.argmax([np.abs(c[0]).sum() for c in raw]))][0]
        elif np.ndim(raw) == 3:
            values = raw[0, :, int(np.argmax(np.abs(raw[0]).sum(axis=0)))]
        else:
            values = raw[0]
        pairs = sorted(zip(feature_names, values), key=lambda p: abs(p[1]), reverse=True)
        return pairs[:top_k]

    def explain_lstm_gradient(model, token_idx_seq, target_class_idx):
        """DeepSHAP/GradientExplainer for the BiLSTM, with a manual
        tf.GradientTape saliency fallback. shap's TF integration
        (DeepExplainer/GradientExplainer) is known to be fragile against
        newer TF2/Keras3 eager-mode graphs -- verified failing in this
        environment, so we try shap first (for portability to environments
        where it does work) and fall back to a directly-computed
        Gradient x Input attribution, which is the same idea DeepSHAP
        approximates."""
        x = np.array([token_idx_seq])
        try:
            explainer = shap.GradientExplainer(model, x)
            sv = explainer.shap_values(x)
            values = sv[0][..., target_class_idx] if isinstance(sv, list) else sv[..., target_class_idx][0]
            return values.tolist()
        except Exception:
            pass
        embed_layer = model.layers[0]
        emb = embed_layer(x)
        with tf.GradientTape() as tape:
            tape.watch(emb)
            h = model.layers[1](emb)
            logits = model.layers[2](h)
            target = logits[:, target_class_idx]
        grad = tape.gradient(target, emb)
        attribution = tf.reduce_sum(grad * emb, axis=-1).numpy()[0]
        return attribution.tolist()

    attack_sessions = test_sessions[test_sessions["alert_tier"] == "ATTACK"].head(3)
    if attack_sessions.empty:
        print("No ATTACK-tier sessions in this run/dataset -- nothing to explain.")
    else:
        for _, row in attack_sessions.iterrows():
            sid = row["session_id"]
            flows = test_scaled[test_scaled["session_id"] == sid]
            predicted_idx = CLASS_TO_IDX[row["predicted_class"]]

            rf_top = explain_tree_model(rf_branchA, flows[rf_cols].iloc[[0]], rf_cols)
            xgb_top = explain_tree_model(xgb_branchA, flows[xgb_cols].iloc[[0]], xgb_cols)

            tok_seq = row["tokens"]
            tok_idx = [TOKEN_TO_IDX[t] + 1 for t in tok_seq][:MAX_LEN]
            padded = tok_idx + [0] * (MAX_LEN - len(tok_idx))
            lstm_attrib = explain_lstm_gradient(bilstm, padded, predicted_idx)[: len(tok_idx)]

            path = [(t1, t2, edge_weight(t1, t2)) for t1, t2 in zip(tok_seq, tok_seq[1:])]

            print(f"\n=== ATTACK evidence chain: session {sid} ===")
            print(f"  True label: {row['label']}  |  Predicted: {row['predicted_class']}  |  Risk_t={row['Risk_t']:.3f}")
            print(f"  Token path: {' -> '.join(tok_seq)}")
            print(f"  Graph edges: {[(a,b,round(w,3)) for a,b,w in path]}")
            print(f"  RF top-5 features: {[(f, round(v,4)) for f, v in rf_top]}")
            print(f"  XGBoost top-5 features: {[(f, round(v,4)) for f, v in xgb_top]}")
            print(f"  LSTM token attributions: {[(t, round(a,5)) for t, a in zip(tok_seq, lstm_attrib)]}")

except Exception as e:
    print(f"[Cell 13 ERROR] Stage 12 failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 14 -- Stage 13: Evaluation
# ============================================================
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, confusion_matrix,
    roc_auc_score, accuracy_score,
)

try:
    y_test = test_sessions["label"].values

    def false_positive_rate(y_true, y_pred, negative_label="BENIGN"):
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        neg = y_true == negative_label
        fp = np.sum(neg & (y_pred != negative_label))
        tn = np.sum(neg & (y_pred == negative_label))
        return float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0

    def safe_roc_auc(y_true_idx, proba, classes=CLASSES):
        present = sorted(set(y_true_idx))
        try:
            y_bin = np.eye(len(classes))[y_true_idx][:, present]
            return roc_auc_score(y_bin, proba[:, present], average="macro", multi_class="ovr")
        except Exception:
            return float("nan")

    y_test_idx = np.array([CLASS_TO_IDX[l] for l in y_test])

    proba_rf_test_branchB_sess = flow_proba_to_session(test_scaled, proba_rf_test_branchB).reindex(test_sessions["session_id"]).values

    model_predictions = {
        "Fused (RF+LSTM+XGB)": test_sessions["predicted_class"].values,
        "Standalone RF (Branch A)": np.array([CLASSES[i] for i in P_A_test_sess.argmax(1)]),
        "Standalone XGBoost (Branch A)": np.array([CLASSES[i] for i in P_C_test_sess.argmax(1)]),
        "Standalone BiLSTM": np.array([CLASSES[i] for i in P_B_test_sess.argmax(1)]),
    }
    model_probas = {
        "Fused (RF+LSTM+XGB)": R_t_test, "Standalone RF (Branch A)": P_A_test_sess,
        "Standalone XGBoost (Branch A)": P_C_test_sess, "Standalone BiLSTM": P_B_test_sess,
    }

    summary_rows = []
    for name, y_pred in model_predictions.items():
        summary_rows.append({
            "model": name,
            "macro_f1": f1_score(y_test, y_pred, labels=CLASSES, average="macro", zero_division=0),
            "accuracy": accuracy_score(y_test, y_pred),
            "fpr": false_positive_rate(y_test, y_pred),
            "roc_auc_macro_ovr": safe_roc_auc(y_test_idx, model_probas[name]),
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
    print("=== Model comparison (TEST sessions) ===")
    print(summary_df.to_string(index=False))

    best_name = summary_df.iloc[0]["model"]
    y_pred_best = model_predictions[best_name]

    precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_best, labels=CLASSES, zero_division=0)
    per_class_df = pd.DataFrame({"class": CLASSES, "precision": precision, "recall": recall, "f1": f1, "support": support})
    print(f"\n=== Per-class F1 table ({best_name}) ===")
    print(per_class_df.to_string(index=False))

    print(f"\n=== Confusion matrix ({best_name}) ===")
    cm = pd.DataFrame(confusion_matrix(y_test, y_pred_best, labels=CLASSES), index=CLASSES, columns=CLASSES)
    print(cm)

    # McNemar's test: Branch A vs Branch B RF predictions on the SAME test
    # sessions -- does SMOTE+ENN resampling change which sessions RF gets
    # right, relative to class-weighting alone?
    y_pred_rf_branchA = model_predictions["Standalone RF (Branch A)"]
    y_pred_rf_branchB = np.array([CLASSES[i] for i in proba_rf_test_branchB_sess.argmax(1)])
    correct_A = y_pred_rf_branchA == y_test
    correct_B = y_pred_rf_branchB == y_test
    b = int(np.sum(correct_A & ~correct_B))
    c = int(np.sum(~correct_A & correct_B))
    if b + c == 0:
        mcnemar_stat, mcnemar_p = 0.0, 1.0
    elif b + c < 25:
        mcnemar_p = stats.binomtest(min(b, c), b + c, 0.5).pvalue
        mcnemar_stat = float(min(b, c))
    else:
        mcnemar_stat = (abs(b - c) - 1) ** 2 / (b + c)
        mcnemar_p = 1 - stats.chi2.cdf(mcnemar_stat, df=1)
    print(f"\nMcNemar's test (RF Branch A vs Branch B): statistic={mcnemar_stat:.3f}, p={mcnemar_p:.4f} "
          f"(A-right/B-wrong={b}, A-wrong/B-right={c})")

    # Bootstrap 95% CI on the best model's macro-F1 (case resampling, n=1000).
    rng = np.random.default_rng(RANDOM_SEED)
    n = len(y_test)
    boot_scores = np.empty(1000)
    for i in range(1000):
        idx = rng.integers(0, n, size=n)
        boot_scores[i] = f1_score(y_test[idx], y_pred_best[idx], labels=CLASSES, average="macro", zero_division=0)
    ci_lo, ci_hi = np.percentile(boot_scores, [2.5, 97.5])
    print(f"\nBootstrap 95% CI on macro-F1 ({best_name}): "
          f"{summary_df.iloc[0]['macro_f1']:.4f} [{ci_lo:.4f}, {ci_hi:.4f}] (n=1000 resamples)")

except Exception as e:
    print(f"[Cell 14 ERROR] Stage 13 failed: {type(e).__name__}: {e}")
    raise
